In [ ]:
import ipywidgets as widgets
from IPython.display import display
import os

box_layout = widgets.Layout(
    display="flex",
    flex_flow="column",
    align_items="center",
    border="1px solid #E0E0E0",
    width="400px",
    padding="25px",
    border_radius="10px"
)

button_layout = widgets.Layout(width="100%", height="40px")

upload_layout = widgets.Layout(width="100%")

title = widgets.HTML(
    "<h3 style='margin-bottom:10px;'>Upload CSV Files 👇</h3>"
)

subtitle = widgets.HTML(
    "<p style='color:gray; margin-top:0;'>Select one or more CSV files to upload and save</p>"
)

train_uploader = widgets.FileUpload(
    accept='.csv',
    multiple=True,
    layout=upload_layout
)

save_button = widgets.Button(
    description='Save Files',
    button_style='primary',
    icon='save',
    layout=button_layout
)

status = widgets.HTML("<p style='color:gray;'>Waiting for upload...</p>")

output = widgets.Output()

def save_files(change=None):
    with output:
        output.clear_output()
        
        if train_uploader.value:
            saved_files = []
            for filename, file_info in train_uploader.value.items():
                with open(filename, "wb") as f:
                    f.write(file_info['content'])
                saved_files.append(filename)
            
            status.value = f"<p style='color:green;'>Saved {len(saved_files)} file(s) 👍</p>"
        else:
            status.value = "<p style='color:red;'>⚠️ No files uploaded yet</p>"

train_uploader.observe(save_files, names='value')

save_button.on_click(save_files)

card = widgets.VBox(
    [title, subtitle, train_uploader, save_button, status],
    layout=box_layout
)

display(card)
display(output)

Output()

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

import seaborn as sb

In [4]:
df = pd.read_csv('file.csv')[['Age','Pclass','SibSp','Parch','Survived']]

In [5]:
df.sample(3)

,Age,Pclass,SibSp,Parch,Survived
664,20.0,3,1,0,1
778,NaN,3,0,0,0
599,49.0,1,1,0,1


In [6]:
df.dropna(inplace=True)

In [7]:
df.isnull().sum()

,0
Age,0
Pclass,0
SibSp,0
Parch,0
Survived,0


In [8]:
x = df.drop(columns=['Survived'])
y = df['Survived']

In [9]:
x.sample(2)

,Age,Pclass,SibSp,Parch
156,16.0,3,0,0
647,56.0,1,0,0


In [11]:
np.mean(cross_val_score(
    estimator = LogisticRegression(),
    X= x,
    y= y,
    scoring='accuracy',
    cv=20
))

np.float64(0.6933333333333332)

### **Feature construction**

In [12]:
x['Family_size'] = x['SibSp'] + x['Parch'] + 1

In [13]:
x.sample(2)

,Age,Pclass,SibSp,Parch,Family_size
338,45.0,3,0,0,1
307,17.0,1,1,0,2


In [14]:
def family_type(family_size):
    if family_size == 1:    # alone
        return 0
    elif family_size > 1 and family_size < 4:       # small family
        return 1
    else:
        return 2

In [15]:
x['Family_Type'] = x['Family_size'].apply(family_type)

In [20]:
x.sample(2)

,Age,Pclass,SibSp,Parch,Family_size,Family_Type
289,22.0,3,0,0,1,0
787,8.0,3,4,1,6,2


In [21]:
x.drop(columns=['SibSp','Parch','Family_size'],inplace=True)

In [22]:
x.sample(5)

,Age,Pclass,Family_Type
331,45.5,1,0
99,34.0,2,1
21,34.0,2,0
242,29.0,2,0
434,50.0,1,1


### **Result**

In [23]:
np.mean(cross_val_score(
    estimator = LogisticRegression(),
    X= x,
    y= y,
    scoring='accuracy',
    cv=20
))

np.float64(0.7074206349206349)